In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from typing import List

class SmartPDFPreProcessor:

    def __init__(self, chunk_size=200, chunk_overlap=20):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.text_splitter = RecursiveCharacterTextSplitter(separators=[" "],chunk_size=chunk_size,chunk_overlap=chunk_overlap)

    def preprocess(self, pdf_path) -> List[dict]:
        # load the pdf
        pages = PyPDFLoader(pdf_path).load()
        processed_chunks = []
        # clean the pages
        for page_num, page in enumerate(pages):
            clean_text = self._clean_text(page.page_content)
            # remove empty pages
            if len(clean_text.strip()) < 50:
                continue
            chunks = self.text_splitter.create_documents(
                texts=[clean_text],
                metadatas=[{
                    **page.metadata,
                    "page": page_num + 1,
                    "total_pages": len(pages),
                    "chunk_method": "smart pdf processor",
                    "char_count": len(clean_text)
                }]
            )
            processed_chunks.extend(chunks)
        return processed_chunks

    def _clean_text(self, text):
        # remove new lines and extra spaces
        text = text.replace("\n", " ")
        text = " ".join(text.split())
        return text

In [10]:
try:
    smart_chunks = SmartPDFPreProcessor().preprocess("data/pdf/attention.pdf")
    print(f"Processed {len(smart_chunks)} chunks from the PDF.")
    print("Sample chunk metadata:", smart_chunks[0].metadata if smart_chunks else "No chunks processed.")
    print("Sample chunk content:", smart_chunks[0].page_content if smart_chunks else "No chunks processed.")
except Exception as e:
    print(f"An error occurred while processing the PDF: {e}") 


Processed 225 chunks from the PDF.
Sample chunk metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/pdf/attention.pdf', 'total_pages': 15, 'page': 1, 'page_label': '1', 'chunk_method': 'smart pdf processor', 'char_count': 2855}
Sample chunk content: Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need
